# Relative Approximation Ratio across 5 Initialization Strategies

Optimization performance (relative approximation ratio $r_{RA}$) of IQP circuits
for three connectivity ansaetze (single / circular / full) across three
Hamiltonians (Ising / MaxCut / Partition) and **five** parameter initializations.
This is the Fig. 3 counterpart of the gradient-variance benchmark
(`GV_5_Initializations.ipynb`).

| key | init | distribution |
|-----|------|--------------|
| `normal`  | Gaussian        | N(0, 1) |
| `uniform` | Uniform         | U(-pi, pi) |
| `pi4`     | Near-pi/4       | U(pi/4 - eps, pi/4 + eps), eps = 0.05 |
| `he`      | He (Kaiming)    | N(0, 2/n) |
| `glorot`  | Glorot (Xavier) | N(0, 1/n) |

Each config is optimized with IQPopt (Adam, `n_iters` iterations, `n_samples`
Monte Carlo samples) over `n_problem` random instances. The only thing that
changes across the five initializations is the sampled `params_init` vector.
The raw per-instance ratios are stored in `ratio_5inits.json`.

In [1]:
import json
import numpy as np
import jax
import matplotlib.pyplot as plt

from IQP_circuit import *          # iqp, gens, loss_fn, train_model
from MaxCut import *               # maxcut_value, solve_maxcut_exact_symmetry
from Partition import *            # sum_selected
from numberpartitioning import karmarkar_karp

## Initialization sampler and Ising observable
`init_params` returns the flat `params_init` vector (length = number of IQP
generators) for each of the five strategies. He/Glorot use the circuit width
`n` as the fan (convention A: variance 2/n and 1/n respectively).

In [2]:
PI4_EPS = 0.05  # half-width of the near-pi/4 window

def init_params(name, n_gates, n_qubits):
    if name == 'normal':
        return np.random.normal(0.0, 1.0, n_gates)
    if name == 'uniform':
        return np.random.uniform(-np.pi, np.pi, n_gates)
    if name == 'pi4':
        return np.random.uniform(np.pi/4 - PI4_EPS, np.pi/4 + PI4_EPS, n_gates)
    if name == 'he':
        return np.random.normal(0.0, np.sqrt(2.0 / n_qubits), n_gates)
    if name == 'glorot':
        return np.random.normal(0.0, np.sqrt(1.0 / n_qubits), n_gates)
    raise ValueError(f"Unknown init '{name}'")

def ising_obs_local(n_qubits, corr=2):
    obs = []
    for i in range(n_qubits):
        v = [0] * n_qubits; v[i] = 1; obs.append(v)
    if corr == 2:
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                v = [0] * n_qubits; v[i] = 1; v[j] = 1; obs.append(v)
    return np.array(obs)

## Load benchmark instances and exact reference values

In [3]:
with open("ising.json", "r") as f:
    dataset_ising = {int(k): v for k, v in json.load(f).items()}
with open("ising_loss_exact.json", "r") as f:
    exact_ising = {int(k): v for k, v in json.load(f).items()}
with open("maxcut.json", "r") as f:
    dataset_maxcut = {int(k): v for k, v in json.load(f).items()}
with open("partition.json", "r") as f:
    dataset_partition = {int(k): v for k, v in json.load(f).items()}

print("Datasets loaded successfully")

Datasets loaded successfully


## Relative approximation ratio estimators (one per Hamiltonian)
Each estimator trains one IQP circuit and returns $r_{RA}$ for a single
(instance, connectivity, initialization). The ratio definitions are identical
to the original per-Hamiltonian optimization notebooks:
- **Ising**: $(f_{\text{exact}} - f(\theta)) / f_{\text{exact}}$ from the final loss.
- **MaxCut**: $|f - f^*| / f^*$ using the decoded bitstring and an exact solver.
- **Partition**: $|f - f^*| / f^*$ using Karmarkar--Karp as the reference.

In [4]:
N_ITERS   = 1000   # optimization steps (Adam)
N_SAMPLES = 1000   # Monte Carlo samples per expectation
KEY       = jax.random.PRNGKey(42)

def _train(ops, coeffs, gates, n_qubits, params_init):
    circuit = iqp.IqpSimulator(n_qubits, gates)
    trainer = train_model(circuit=circuit, ops=ops, coeffs=coeffs,
                          params_init=params_init, key=KEY, loss_fn=loss_fn,
                          optimizer="Adam", stepsize=1e-3,
                          n_iters=N_ITERS, n_samples=N_SAMPLES)
    return circuit, trainer

def ratio_ising(n, j, init, mode_circuit):
    coeffs = np.array(dataset_ising[n][j])
    ops = ising_obs_local(n, 2)
    gates = gens(n, mode_circuit)
    p = init_params(init, len(gates), n)
    _, tr = _train(ops, coeffs, gates, n, p)
    return float((exact_ising[n][j] - tr.losses[-1]) / exact_ising[n][j])

def ratio_maxcut(n, j, init, mode_circuit):
    edges, weights = dataset_maxcut[n][j]
    ops, coeffs = maxcut_obs(n, edges, weights)
    gates = gens(n, mode_circuit)
    p = init_params(init, len(gates), n)
    circ, tr = _train(ops, coeffs, gates, n, p)
    probs = circ.probs(tr.final_params)
    bits = np.binary_repr(int(np.argmax(probs)), n)
    sol = maxcut_value(bits, edges, weights)
    ex = solve_maxcut_exact_symmetry(edges, weights)[0]
    return float(abs(sol - ex) / ex)

def ratio_partition(n, j, init, mode_circuit):
    numbers = dataset_partition[n][j]
    ops, coeffs = number_partition_obs(numbers)
    gates = gens(n, mode_circuit)
    p = init_params(init, len(gates), n)
    circ, tr = _train(ops, coeffs, gates, n, p)
    probs = circ.probs(tr.final_params)
    bits = np.binary_repr(int(np.argmax(probs)), n)
    sol = sum_selected(bits, numbers)
    ex = karmarkar_karp(numbers, num_parts=2)
    denom = sum(ex.partition[0])
    return float(abs(sol - denom) / denom)

ratio_fns = {'ising': ratio_ising, 'maxcut': ratio_maxcut, 'partition': ratio_partition}

## Compute approximation ratio for all 5 initializations
**Heavy:** `5 inits x 3 Hamiltonians x 3 circuits x len(qubits) x n_problem`
optimization runs of `N_ITERS` steps each. For a quick end-to-end check, reduce
`qubits`, `n_problem`, and `N_ITERS` first (e.g. `qubits=[3,6]`, `n_problem=5`,
`N_ITERS=200`). Stored **raw** per-instance ratios.

In [5]:
inits             = ['normal', 'uniform', 'pi4', 'he', 'glorot']
mode_hamiltonians = ['ising', 'maxcut', 'partition']
mode_circuits     = ['single', 'circular', 'full']
qubits            = [3, 6, 9, 12, 15, 18]
n_problem         = 50

all_results = {}

for init_name in inits:
    print(f"\n{'='*40}\nInit: {init_name}")
    results = {}
    for mode_ham in mode_hamiltonians:
        print(f"  === {mode_ham} ===")
        results[mode_ham] = {}
        rfn = ratio_fns[mode_ham]
        for mode_circuit in mode_circuits:
            print(f"    {mode_circuit}")
            results[mode_ham][mode_circuit] = {}
            for n_qubits in qubits:
                print(f"      n={n_qubits}", end=" ... ", flush=True)
                ratios = [rfn(n_qubits, j, init_name, mode_circuit)
                          for j in range(n_problem)]
                results[mode_ham][mode_circuit][n_qubits] = ratios
                print(f"mean={np.mean(ratios):.3f}")
    all_results[init_name] = results

with open("ratio_5inits.json", "w") as f:
    json.dump(all_results, f, indent=4)
print("\nSaved to ratio_5inits.json")


Init: normal
  === ising ===
    single
      n=3 ... 

Training Progress:  36%|███▌      | 358/1000 [00:09<00:16, 37.97it/s, loss=-1.147344, elapsed time=0.02, total time=10]  


KeyboardInterrupt: 

## Plot: 3 Hamiltonians x 5 initializations (Fig. 3)
Linear $y$-axis with **shared per-row limits** (Ising $[0,1]$, MaxCut $[0,0.6]$,
Number Partition $[0,0.4]$); $y$-ticks shown only on the first column. Lower
$r_{RA}$ = better optimization.

In [ ]:
with open("ratio_5inits.json", "r") as f:
    all_results = json.load(f)

qubits = [3, 6, 9, 12, 15, 18]

ham_names = {'ising': 'Classical Ising', 'maxcut': 'MaxCut', 'partition': 'Number Partition'}
styles = {'full': ('o-', '#1f77b4'), 'circular': ('s-', '#ff7f0e'), 'single': ('^--', '#2ca02c')}
labels = {'full': 'Full Connectivity', 'circular': 'Circular Connectivity', 'single': 'Single-Z Terms'}

init_cols  = ['normal', 'uniform', 'pi4', 'he', 'glorot']
col_titles = {
    'normal':  r'$\mathcal{N}(0, 1)$',
    'uniform': r'$\mathcal{U}(-\pi, \pi)$',
    'pi4':     r'$\pi/4$ perturbation',
    'he':      'He',
    'glorot':  'Glorot',
}
row_ylim = {0: (0.0, 1.0), 1: (0.0, 0.6), 2: (0.0, 0.4)}

fig, axes = plt.subplots(3, 5, figsize=(20, 11))

for row, mode_ham in enumerate(['ising', 'maxcut', 'partition']):
    for col, init_name in enumerate(init_cols):
        ax  = axes[row][col]
        res = all_results[init_name][mode_ham]

        for mode_circuit in ['full', 'circular', 'single']:
            means = np.array([np.mean(res[mode_circuit][str(n)]) for n in qubits])
            stds  = np.array([np.std(res[mode_circuit][str(n)])  for n in qubits])
            fmt, color = styles[mode_circuit]
            ax.plot(qubits, means, fmt, color=color, label=labels[mode_circuit],
                    linewidth=2, markersize=5)
            ax.fill_between(qubits, means - stds, means + stds, color=color, alpha=0.2)

        ax.set_ylim(*row_ylim[row])
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xlabel('Number of Qubits')
        if col == 0:
            ax.set_ylabel(f'{ham_names[mode_ham]}\n'
                          r'Relative approx. ratio $r_{RA}$', fontsize=10)
        else:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(col_titles[init_name], fontsize=13)

handles, labels_ = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels_, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 1.02), frameon=False, fontsize=11)

plt.tight_layout()
plt.savefig("Ratio_5_initializations.pdf", dpi=300, bbox_inches='tight')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'ratio_5inits.json'